# Qualify the MNIST JointAE-32 metric encoder

This notebook explains and reproduces the standalone encoder training job.

The official training split has two roles. A seeded 55,000-image subset updates weights. The remaining 5,000 images select one checkpoint.

The official test split never updates weights or selects a checkpoint. It measures fixed-weight performance and defines MNIST reference statistics.

The raw 128-dimensional linear projection defines MNIST JointAE FD. Feature-space memorization applies separate training-bank normalization.

Run this notebook on an allocated GPU. The public metric package does not depend on this notebook.

In [ ]:
# Set the exact training controls and locate the repository.
import copy
import os
import runpy
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms.functional import rotate

REPO_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "mem_param_src").is_dir()
)
DATA_ROOT = Path(
    os.environ.get("MP_MNIST_DATA_ROOT", REPO_ROOT.parent / "old_mem_par/data")
)
DEVICE = torch.device(os.environ.get("MP_NOTEBOOK_DEVICE", "cuda"))
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Request an allocated GPU or set MP_NOTEBOOK_DEVICE=cpu.")

SEED = 2003
EPOCHS = 10
BATCH_SIZE = 256
CLASSIFICATION_WEIGHT = 0.1
MINIMUM_VALIDATION_ACCURACY = 0.99

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"repository: {REPO_ROOT}")
print(f"data root: {DATA_ROOT}")
print(f"device: {DEVICE}")

## Load the frozen training definition

The standalone job owns the optimizer, split, architecture, and training loop. Loading it here prevents notebook drift.

The source package contains only the frozen evaluation implementation.

In [ ]:
# Load the standalone trainer without executing its command-line entrypoint.
trainer = runpy.run_path(
    REPO_ROOT / "jobs/berzelius/mnist_jointae32/train.py",
    run_name="mnist_jointae_notebook",
)
MNISTTensorDataset = trainer["MNISTTensorDataset"]
MNISTJointAE32 = trainer["MNISTJointAE32"]
canonicalize_mnist32 = trainer["canonicalize_mnist32"]
collect_features = trainer["collect_features"]
is_better_candidate = trainer["is_better_candidate"]
make_loader = trainer["make_loader"]
run_epoch = trainer["run_epoch"]

model = MNISTJointAE32(feature_dim=128).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
print(f"parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")
print(f"raw feature dimension: {model.feature_dim}")

In [ ]:
# Build the deterministic 55k training, 5k validation, and 10k test partitions.
train_source = datasets.MNIST(DATA_ROOT, train=True, download=False)
test_source = datasets.MNIST(DATA_ROOT, train=False, download=False)
train_dataset = MNISTTensorDataset(train_source)
test_dataset = MNISTTensorDataset(test_source)

# Freeze the training-split permutation with the encoder seed.
generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_dataset), generator=generator)
train_indices = permutation[:55_000].tolist()
valid_indices = permutation[55_000:].tolist()
test_indices = list(range(10_000))

# Shuffle only the optimization loader.
train_loader = make_loader(
    train_dataset, train_indices, BATCH_SIZE, 4, True, SEED + 1, DEVICE
)
valid_loader = make_loader(
    train_dataset, valid_indices, BATCH_SIZE, 4, False, SEED + 2, DEVICE
)
test_loader = make_loader(
    test_dataset, test_indices, BATCH_SIZE, 4, False, SEED + 3, DEVICE
)
print(f"train/validation/test: {len(train_indices)}/{len(valid_indices)}/{len(test_indices)}")

## Train and restore one qualified checkpoint

Each epoch records online training metrics and fixed-weight validation metrics.

The selected checkpoint has the lowest validation reconstruction loss among epochs with at least 99% validation accuracy.

In [ ]:
# Train for ten epochs and retain the qualified validation checkpoint.
history = []
best_epoch = None
best_validation = None
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        DEVICE,
        CLASSIFICATION_WEIGHT,
        optimizer=optimizer,
    )
    validation_metrics = run_epoch(
        model,
        valid_loader,
        DEVICE,
        CLASSIFICATION_WEIGHT,
    )
    history.append(
        {"epoch": epoch, "train": train_metrics, "validation": validation_metrics}
    )
    if is_better_candidate(
        validation_metrics,
        best_validation,
        MINIMUM_VALIDATION_ACCURACY,
    ):
        best_epoch = epoch
        best_validation = dict(validation_metrics)
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"epoch {epoch:02d}: "
        f"train recon={train_metrics['reconstruction_loss']:.5f}, "
        f"valid recon={validation_metrics['reconstruction_loss']:.5f}, "
        f"valid accuracy={validation_metrics['accuracy']:.4f}"
    )

# Restore weights before every fixed-weight evaluation.
model.load_state_dict(best_state)
model.eval()
print(f"selected epoch: {best_epoch}")
print(best_validation)

In [ ]:
# Recompute train, validation, and test metrics with the selected weights.
fixed_train_loader = make_loader(
    train_dataset, train_indices, BATCH_SIZE, 4, False, SEED + 4, DEVICE
)
fixed_metrics = {
    "train": run_epoch(
        model, fixed_train_loader, DEVICE, CLASSIFICATION_WEIGHT
    ),
    "validation": run_epoch(
        model, valid_loader, DEVICE, CLASSIFICATION_WEIGHT
    ),
    "test": run_epoch(
        model, test_loader, DEVICE, CLASSIFICATION_WEIGHT
    ),
}
for partition, metrics in fixed_metrics.items():
    print(
        f"{partition:10s}: recon={metrics['reconstruction_loss']:.6f}, "
        f"class={metrics['classification_loss']:.6f}, "
        f"accuracy={metrics['accuracy']:.4f}"
    )

In [ ]:
# Show fixed-weight reconstructions and classifier predictions.
images, labels = next(iter(valid_loader))
images = canonicalize_mnist32(images.to(DEVICE))
with torch.no_grad():
    reconstructions, logits, _ = model(images)
predictions = logits.argmax(1)

figure, axes = plt.subplots(2, 10, figsize=(14, 3))
for index in range(10):
    axes[0, index].imshow(images[index, 0].cpu(), cmap="gray", vmin=0, vmax=1)
    axes[0, index].set_title(f"y={int(labels[index])}")
    axes[1, index].imshow(
        reconstructions[index, 0].cpu(), cmap="gray", vmin=0, vmax=1
    )
    axes[1, index].set_title(f"p={int(predictions[index])}")
    axes[0, index].axis("off")
    axes[1, index].axis("off")
figure.suptitle("Validation images and selected-checkpoint reconstructions")
plt.tight_layout()
plt.show()

## Qualify raw-feature Fréchet distance

The next cells use raw encoder features. They do not apply LayerNorm, SiLU, standardization, or L2 normalization.

The clean split estimates the finite-sample floor. Noise, translation, rotation, and reduced class support test metric sensitivity.

In [ ]:
# Collect fixed-weight test features and define one image-query encoder.
from mem_param_src.metrics.image import (
    compute_feature_statistics,
    compute_frechet_distance,
)

test_features, test_labels, test_logits = collect_features(
    model, test_loader, DEVICE
)

@torch.no_grad()
def encode_query(images, batch_size=BATCH_SIZE):
    # Canonicalize and encode every query with the selected weights.
    blocks = []
    for start in range(0, len(images), batch_size):
        batch = canonicalize_mnist32(images[start : start + batch_size].to(DEVICE))
        blocks.append(model.encode(batch).cpu())
    return torch.cat(blocks)

first_half = compute_feature_statistics(test_features[:5_000])
second_half = compute_feature_statistics(test_features[5_000:])
clean_floor = compute_frechet_distance(first_half, second_half)
print(f"disjoint test-half JointAE FD: {clean_floor:.6f}")
print(f"test accuracy from saved logits: {(test_logits.argmax(1) == test_labels).float().mean():.4f}")

In [ ]:
# Measure FD sensitivity to noise, translation, rotation, and class collapse.
probe = torch.stack([test_dataset[index][0] for index in range(2_000)])
probe32 = canonicalize_mnist32(probe)
reference_stats = compute_feature_statistics(test_features)

def fd_for_images(images):
    # Compare one transformed image bank with the fixed official test reference.
    return compute_frechet_distance(
        compute_feature_statistics(encode_query(images)),
        reference_stats,
    )

diagnostics = {"clean": fd_for_images(probe32)}
for sigma in (0.05, 0.10, 0.20, 0.40):
    noisy = (probe32 + sigma * torch.randn_like(probe32)).clamp(0, 1)
    diagnostics[f"noise_{sigma:.2f}"] = fd_for_images(noisy)

# Translate without wraparound pixels.
translated = F.pad(probe32, (3, 0, 3, 0))[:, :, :32, :32]
diagnostics["translation_3px"] = fd_for_images(translated)
rotated = rotate(probe32, angle=15.0, interpolation=2)
diagnostics["rotation_15deg"] = fd_for_images(rotated)

for class_count in (10, 5, 2, 1):
    mask = test_labels < class_count
    diagnostics[f"class_support_{class_count}"] = compute_frechet_distance(
        compute_feature_statistics(test_features[mask]),
        reference_stats,
    )

for name, value in diagnostics.items():
    print(f"{name:20s}: {value:.6f}")

## Demonstrate feature-space memorization

JointAE memorization uses the evaluated generative run's exact training bank.

The code standardizes coordinates with that bank, then L2-normalizes each vector. It computes both nearest neighbors once.

Held-out ratios calibrate four thresholds. Generated queries never select their own threshold.

In [ ]:
# Demonstrate d1, d2, r, and f_mem with one query abstraction.
from mem_param_src.metrics.image import compute_mnist_feature_memorization

# Keep this notebook demonstration small enough for interactive inspection.
metric_training = collect_features(
    model,
    make_loader(train_dataset, train_indices[:2_048], BATCH_SIZE, 4, False, SEED + 5, DEVICE),
    DEVICE,
)[0]
heldout_query = test_features[:512]
calibration_record = compute_mnist_feature_memorization(
    heldout_query,
    metric_training,
    rho=0.3,
    query_batch_size=256,
    reference_batch_size=512,
)
alphas = (0.005, 0.01, 0.02, 0.05)
thresholds = {
    alpha: float(torch.quantile(calibration_record["r"], alpha))
    for alpha in alphas
}

query_features = {
    "exact_training_copies": metric_training[:128].clone(),
    "heldout": test_features[512:640].clone(),
    "noisy_heldout": encode_query(
        (probe32[512:640] + 0.20 * torch.randn_like(probe32[512:640])).clamp(0, 1)
    ),
}

query_records = {}
for name, features in query_features.items():
    record = compute_mnist_feature_memorization(
        features,
        metric_training,
        rho=thresholds[0.01],
        query_batch_size=128,
        reference_batch_size=512,
    )
    query_records[name] = record
    print(
        f"{name:22s}: f_mem={record['f_mem']:.4f}, "
        f"median d1={record['d1'].median():.5f}, "
        f"median d2={record['d2'].median():.5f}, "
        f"median r={record['r'].median():.5f}"
    )

copy_record = query_records["exact_training_copies"]
assert torch.allclose(copy_record["d1"], torch.zeros_like(copy_record["d1"]), atol=1e-6)
assert copy_record["f_mem"] > 0.99
print(f"rho thresholds: {thresholds}")

In [ ]:
# Plot the qualification diagnostics from saved numeric records.
noise_names = [name for name in diagnostics if name.startswith("noise_")]
noise_levels = [float(name.split("_")[1]) for name in noise_names]
noise_values = [diagnostics[name] for name in noise_names]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(noise_levels, noise_values, marker="o")
axes[0].axhline(clean_floor, color="black", linestyle="--", label="clean split floor")
axes[0].set_xlabel("Gaussian noise standard deviation")
axes[0].set_ylabel("MNIST JointAE FD")
axes[0].legend()

support_counts = [10, 5, 2, 1]
support_values = [diagnostics[f"class_support_{count}"] for count in support_counts]
axes[1].plot(support_counts, support_values, marker="o")
axes[1].invert_xaxis()
axes[1].set_xlabel("Retained digit classes")
axes[1].set_ylabel("MNIST JointAE FD")
plt.tight_layout()
plt.show()

## Use the packaged metric

The packaged loader verifies the bundled checkpoint hash. It returns raw 128-dimensional features.

```python
from mem_param_src.metrics.image import load_mnist_jointae32

encoder = load_mnist_jointae32(device="cuda")
features = encoder(images, image_range="-1_1")
# features.shape == [batch, 128]
```

Use `mem-param-image-eval` for saved generative runs. The evaluator reads saved tensors, checkpoints, and noise banks.

Do not compare MNIST JointAE FD with Inception FID. They use different representations and answer different dataset-specific questions.